# MNF dimensionality and PPI candidates

This notebook examines the archived dimensionality estimates and verifies the candidate inventory. It checks vote conservation in every processed class–date combination. To recalculate the candidates from observations, use the complete terminal route in notebook 00.

In [ ]:
from pathlib import Path
import os,sys,json
ROOT = next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'config.json').exists())
sys.path.insert(0,str(ROOT))
WORK = Path(os.environ.get('RBC_OUTPUT',ROOT/'work')).resolve()
os.environ['RBC_OUTPUT']=str(WORK)
import pandas as pd
from IPython.display import display,SVG
from dados import CFG


## Estimate noise and count extreme projections

Noise covariance is estimated as half the sample covariance of differences between horizontally or vertically adjacent valid pixels of the same class. The Minimum Noise Fraction (MNF) transform uses this estimate to order spectral components. Noise-Whitened Harsanyi–Farrand–Chang (NWHFC) estimates use a false-alarm probability of 10⁻⁴. The median by class, rounded half up and limited to 2–10, determines how many components are retained.

The Pixel Purity Index (PPI) uses 10,000 unit-length directions generated by normalizing Gaussian random vectors with seed 13. In each projection, the maximum and minimum each receive one vote. Thus the votes sum to 20,000 for every processed class–date combination. All positive-vote candidates retain their original ten-band spectra and locations. The canonical spatial order resolves ties.

In [ ]:
display(json.loads((WORK/'dimensions.json').read_text()))
files=sorted((WORK/'candidates').glob('13-*.parquet'))
quality=pd.read_parquet(WORK/'nwhfc.parquet')
ready=quality[quality.state.eq('ready')]
expected={f'13-{r.date}-{r.classe}.parquet' for r in ready.itertuples()}
assert len(expected)==519 and {p.name for p in files}==expected, 'PPI inventory is incomplete'
voted=0
for p in files:
    f=pd.read_parquet(p);assert f.votes.sum()==20000 and f.votes.gt(0).all()
    voted+=len(f)
assert voted==213647
print(len(files),'class–date combinations verified')
display(pd.read_parquet(WORK/'nwhfc.parquet').groupby('classe').suggested_endmembers.agg(['min','median','max']))


## Read the dimensionality and vote checks

The dimensionality estimates range from two to nine, and the median is six in every class. The 519 processable combinations contain 213,647 positive-vote candidates. The human labeling panel presents up to four candidates per combination, chosen by PPI rank and angular separation; that display sample is distinct from the six retained MNF components.

## References

Green et al. (1988), MNF: https://doi.org/10.1109/36.3001 . Chang and Du (2004), NWHFC: https://doi.org/10.1109/TGRS.2003.819189 . Boardman et al. (1995), PPI: https://ntrs.nasa.gov/citations/19950027316 . Parameters: `METHODOLOGY.md` and `nucleo.py`.